# Metajeux-02 : Casser la composition — où la meilleure réponse composée cesse d'être l'équilibre du jeu composé

**Navigation** : [Version Python](Metajeux-02-Casser-la-Composition-Python.ipynb) | [Index](README.md)

GT-18 a posé la composition d'open games et vérifié la propriété de Hedges — « la meilleure réponse de la composition est la composition des meilleures réponses » — sur un jouet où elle tient, puis mesuré la convergence de la boucle vers son point fixe. Ce notebook fait ce que le critère de maturation demande : **il essaie de casser le jouet, et regarde ce qui casse**.

Deux surfaces d'attaque :

- **Surface A — la sélection n'est pas compositionnelle.** Quand une meilleure réponse locale est *multivoque* (plateau d'optima) et que les paiements sont *couplés*, composer des sélections d'argmax locaux produit un profil qui n'est pas un équilibre du jeu composé. La propriété de Hedges, dans sa forme exacte, porte sur les *correspondances* d'ensembles ; la lecture fonctionnelle du jouet GT-18 — composer des points — ne survit qu'univoque.
- **Surface B — la frontière de contraction.** La convergence de la boucle GT-18 tient parce que l'update est (presque) contractante. En balayant le pas d'update, la boucle passe de la convergence monotone au 2-cycle exact, puis à l'oscillation persistante — et la frontière mesurée coïncide avec la borne de Banach, exactement.

Références : Ghani, N., Hedges, J., Winschel, V., Zahn, P. (2018), *Compositional Game Theory* (LICS '18, arXiv:1603.04641) ; Metajeux-01 (`Metajeux-01-Open-Games-Lentilles-CSharp.ipynb`).

## Objectifs d'apprentissage

1. **Reproduire le régime où la composition tient** : couplée mais univoque, la meilleure réponse composée est bien l'équilibre du composé.
2. **Exhiber la casse** : ajouter UN ingrédient — la multivocité du plateau — et montrer par déviation explicite qu'un profil issu de la composition n'est pas Nash du composé.
3. **Isoler les ingrédients** : le couplage seul ne casse rien (cellule 3), la multivocité seule non plus (r = 0) ; c'est leur *conjonction* qui brise la sélection.
4. **Mesurer la frontière de contraction** : carte convergence / 2-cycle / oscillation sur le pas d'update, comparée à la borne théorique de Banach `L = |1 - delt| < 1`.
5. **Reconsidérer honnêtement le jouet GT-18** : sa boucle additive n'est jamais contractante (L = 1 partout) — sa convergence est une convergence-au-rail, pas un point fixe de Banach.

## 1. Le régime où la composition tient — couplée mais univoque

Rappel du cadre GT-18 : deux open games composés en série. Le joueur 1 choisit `x`, le joueur 2 observe `x` et choisit `y`. Le paiement de J2 dépend de `x` (couplage), celui de J1 de `y` (couplage en retour). Ici chaque meilleure réponse est **univoque** : `u1 = -(x-1)^2` a un unique maximum en `x = 1`, et `u2 = -(y-x)^2` a un unique maximum en `y = x`. La composition des argmax donne le profil `(1, 1)`, et ce profil est bien l'équilibre du jeu composé — vérifié par énumération des déviations, pas par affirmation.

In [1]:
// Illustration 1 — Deux open games en série, COUPLÉS mais UNIVOQUES
using System;

// Composition des sélections : x* = br1(y exogène), puis y* = br2(x*)
int xStar = Metajeux02.Br1(0);
int yStar = Metajeux02.Br2(xStar);

Console.WriteLine($"BR locales univoques : br1(0) = {Metajeux02.Br1(0)}, br2({xStar}) = {yStar}");
Console.WriteLine($"Profil issu de la composition : ({xStar}, {yStar})");
Console.WriteLine($"Ce profil est-il Nash du composé : {Metajeux02.EstNashCompose(xStar, yStar)}");
Console.WriteLine("Régime univoque-couplé : la composition des sélections donne l'équilibre du composé.");

static class Metajeux02
{
    public static readonly int[] ActionsX = { 0, 1, 2 };
    public static readonly int[] ActionsY = { 0, 1, 2 };

    // Paiement J1 : maximum unique en x = 1 (le couplage en y est neutre ici).
    public static double U1(double x, double y) => -(x - 1.0) * (x - 1.0) + 0.0 * y;

    // Paiement J2 : couplée, maximum unique en y = x.
    public static double U2(double x, double y) => -(y - x) * (y - x);

    // Meilleure réponse LOCALE de J1 (y exogène) : univoque.
    public static int Br1(double y)
    {
        int best = ActionsX[0];
        for (int i = 1; i < ActionsX.Length; i++)
        {
            if (U1(ActionsX[i], y) > U1(best, y))
            {
                best = ActionsX[i];
            }
        }
        return best;
    }

    // Meilleure réponse de J2 voyant x : univoque.
    public static int Br2(double x)
    {
        int best = ActionsY[0];
        for (int i = 1; i < ActionsY.Length; i++)
        {
            if (U2(x, ActionsY[i]) > U2(x, best))
            {
                best = ActionsY[i];
            }
        }
        return best;
    }

    // Nash du jeu composé : aucune déviation profitable, J1 anticipant y' = br2(x').
    public static bool EstNashCompose(double x, double y)
    {
        bool ok1 = true;
        foreach (int xp in ActionsX)
        {
            if (!(U1(x, y) >= U1(xp, Br2(xp)) - 1e-12))
            {
                ok1 = false;
            }
        }

        bool ok2 = true;
        foreach (int yp in ActionsY)
        {
            if (!(U2(x, y) >= U2(x, yp) - 1e-12))
            {
                ok2 = false;
            }
        }

        return ok1 && ok2;
    }
}

BR locales univoques : br1(0) = 1, br2(1) = 1


Profil issu de la composition : (1, 1)


Ce profil est-il Nash du composé : True


Régime univoque-couplé : la composition des sélections donne l'équilibre du composé.


**Lecture de la sortie committée** : la composition des argmax produit `(1, 1)`, et l'énumération exhaustive des déviations confirme que c'est l'équilibre du composé. Remarquer ce qui est déjà là : les paiements sont couplés dans les deux sens (`u2` dépend de `x`), et pourtant la composition tient. **Le couplage seul ne casse pas la sélection** — c'est un contre-fait utile : il faut autre chose. C'est cet « autre chose » que la section 2 installe.

## 2. Surface A — la multivocité du plateau brise la sélection

On déplace le maximum de J1 au **centre exact d'un plateau** : `u1 = -(x - 0.5)^2 + r * y` avec `r = 0.3`. Sur `x ∈ {0, 1, 2}`, les valeurs locales sont `-0.25, -0.25, -2.25` — `x = 0` et `x = 1` sont **ex æquo** : la meilleure réponse locale est *multivoque* (`BR = {0, 1}`, plateau par symétrie exacte). La multivocité seule serait inoffensive : avec `r = 0`, J1 resterait indifférent entre les deux profils composés. Mais le couplage `r * y` fait que le choix de J1, via la réponse `y = x` de J2, **revient à J1** : deviner `x = 1` rapporte `r` de plus que `x = 0`. La composition naïve — prendre UN élément du plateau, disons le plus petit — rend un profil que le jeu composé réfute.

In [2]:
// Illustration 2 — Surface A : plateau multivoque + couplage
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

var surfaceA = new Metajeux02SurfaceA();

Console.WriteLine($"Valeurs locales de u1 (y exogène) : [{string.Join(", ", surfaceA.ValeursLocales.Select(v => v.ToString("+0.000;-0.000;0.000", CultureInfo.InvariantCulture)))}]");
Console.WriteLine($"BR locale de J1 = [{string.Join(", ", surfaceA.Br1A)}]  (MULTIVOQUE : plateau par symétrie exacte c=0.5)");

// Composition des sélections : on prend UN élément du plateau (le plus petit)
int selection = surfaceA.Br1A.Min();
var profilsComposes = surfaceA.Br1A.Select(x => (x, Metajeux02.Br2(x))).ToList();
Console.WriteLine($"Profil de la composition naïve (plus petit argmax) : ({selection}, {Metajeux02.Br2(selection)})");
Console.WriteLine($"Ensemble prévu par la composition des correspondances : [{string.Join(", ", profilsComposes.Select(p => $"({p.Item1}, {p.Item2})"))}]");

public class Metajeux02SurfaceA
{
    public const double CPlateau = 0.5;
    public const double RCouplage = 0.3;

    public List<double> ValeursLocales { get; }
    public List<int> Br1A { get; }

    public Metajeux02SurfaceA()
    {
        ValeursLocales = Metajeux02.ActionsX.Select(x => U1A(x, 0.0)).ToList();
        double vmax = ValeursLocales.Max();
        Br1A = Metajeux02.ActionsX
            .Where(x => Math.Abs(U1A(x, 0.0) - vmax) < 1e-12)
            .ToList();
    }

    // Paiement J1 sur la surface A : plateau exact, x = 0 et x = 1 ex aequo en local (c = 0.5 = milieu exact).
    public static double U1A(double x, double y) =>
        -(x - CPlateau) * (x - CPlateau) + RCouplage * y;
}

Valeurs locales de u1 (y exogène) : [-0.250, -0.250, -2.250]


BR locale de J1 = [0, 1]  (MULTIVOQUE : plateau par symétrie exacte c=0.5)


Profil de la composition naïve (plus petit argmax) : (0, 0)


Ensemble prévu par la composition des correspondances : [(0, 0), (1, 1)]


**Lecture de la sortie committée** : la correspondance locale de J1 est `{0, 1}` — le plateau est exact, produit de la symétrie `c = 0.5` au milieu de deux actions consécutives. La composition des correspondances prévoit l'ensemble `{(0,0), (1,1)}`. Reste à demander au jeu composé ce qu'il en pense : chaque profil prédit est confronté à toutes les déviations, J1 anticipant la réponse de J2.

In [3]:
// Illustration 3 — Test Nash du composé, profil par profil
using System;
using System.Globalization;
using System.Linq;

var surfaceA = new Metajeux02SurfaceA();
var profilsComposes = surfaceA.Br1A.Select(x => (x, Metajeux02.Br2(x))).ToList();

double U1A(double x, double y) => Metajeux02SurfaceA.U1A(x, y);

// Écart de paiement si J1 devie vers x' en anticipant y' = br2(x').
double? EcartDeviationJ1(double x, double y)
{
    if (y != Metajeux02.Br2(x))
    {
        return null;   // le profil ne respecte même pas la BR de J2
    }
    return Metajeux02.ActionsX.Max(xp => U1A(xp, Metajeux02.Br2(xp))) - U1A(x, y);
}

Console.WriteLine(string.Format(CultureInfo.InvariantCulture, "{0,-14}{1,9}{2,23}{3,8}  {4}",
    "Profil prevu", "u1(x, y)", "meilleure deviation J1", "ecart", "Verdict"));
Console.WriteLine(new string('-', 78));
foreach (var (x, y) in profilsComposes)
{
    double? ecart = EcartDeviationJ1(x, y);
    int xBest = Metajeux02.ActionsX.OrderByDescending(xp => U1A(xp, Metajeux02.Br2(xp))).First();
    string verdict = (ecart.HasValue && ecart.Value > 1e-9)
        ? "NON-NASH du compose (J1 devie)"
        : "equilibre du compose";
    Console.WriteLine(string.Format(CultureInfo.InvariantCulture, "{0,-14}{1,9:F3}{2,23}{3,8:+0.000;-0.000;0.000}  {4}",
        $"({x}, {y})", U1A(x, y), $"({xBest}, {Metajeux02.Br2(xBest)})", ecart ?? 0.0, verdict));
}

// Contre-fait : R_COUPLAGE = 0 -> le plateau ne casse plus rien
double U1R0(double x, double y) =>
    -(x - Metajeux02SurfaceA.CPlateau) * (x - Metajeux02SurfaceA.CPlateau);
var ecartsR0 = profilsComposes
    .Select(p => Metajeux02.ActionsX.Max(xp => U1R0(xp, Metajeux02.Br2(xp)))
                  - U1R0(p.Item1, Metajeux02.Br2(p.Item1)))
    .ToList();
Console.WriteLine();
Console.WriteLine($"Contre-fait r=0 (plateau sans couplage) : ecarts = [{string.Join(", ", ecartsR0.Select(e => e.ToString("+0.000;-0.000;0.000", CultureInfo.InvariantCulture)))}]"
    + " -> les deux profils survivent (indifference, equilibres faibles)");

Profil prevu   u1(x, y) meilleure deviation J1   ecart  Verdict


------------------------------------------------------------------------------


(0, 0)           -0.250                 (1, 1)  +0.300  NON-NASH du compose (J1 devie)


(1, 1)            0.050                 (1, 1)   0.000  equilibre du compose


Contre-fait r=0 (plateau sans couplage) : ecarts = [0.000, 0.000] -> les deux profils survivent (indifference, equilibres faibles)


**Lecture de la sortie committée** : la casse est exhibée par déviation explicite. Le profil `(0, 0)` — prédit par la composition naïve (sélection du plus petit argmax du plateau) — **n'est pas Nash du composé** : J1, en anticipant que J2 répondra `y = x`, gagne `+0.300` à dévier vers `x = 1`. Seul `(1, 1)` survit. Et le contre-fait `r = 0` confirme l'isolation des ingrédients : sans couplage, le plateau est inoffensif (les deux profils sont des équilibres faibles par indifférence).

**Ce que Hedges garantit vraiment.** La propriété de composition, dans sa forme exacte, porte sur les **correspondances** — les ensembles de meilleures réponses — munies de leur contexte : la composée des correspondances calcule bien l'ensemble des solutions du jeu composé, *à contexte correctement propagé*. Ce qui ne tient pas, et que le jouet univoque de GT-18 ne pouvait pas montrer, est la lecture **fonctionnelle** : composer des *sélections* d'argmax locaux, chacune calculée sans regarder le couplage en retour, ne prédit pas l'équilibre — la composition prédit l'ensemble (sur-approximation), pas le point. Chaque ingrédient est inoffensif seul — couplage sans plateau (section 1), plateau sans couplage (contre-fait r = 0) — c'est leur conjonction qui brise la sélection.

## 3. Surface B — la frontière de contraction de la boucle

La section 3 de GT-18 itère une boucle `W -> P -> decision -> update -> W'` et mesure la distance au point fixe. La famille d'update la plus simple qui généralise la sienne est le **rappel homing** : `a' = clip(a + delt * (b - a), -1, 1)`, qui tire `a` vers la cible `b` avec un pas `delt`. Cette application est affine de pente `1 - delt` (donc de constante de Lipschitz `L = |1 - delt|`) tant que le clip ne mord pas : le **théorème du point fixe de Banach** garantit contraction, donc convergence géométrique vers `b`, si et seulement si `L < 1`, c'est-à-dire `delt ∈ (0, 2)`. À `delt = 2` exactement, `L = 1` : la boucle rebondit de part et d'autre de la cible sans amortir — un 2-cycle exact. Au-delà, chaque pas **amplifie** l'écart (`a - b -> (1-delt)(a - b)` avec `|1-delt| > 1`) jusqu'à atteindre le rail `|a| = 1`, où le clip borne la divergence en un **cycle de période 2 collé au rail** — bornée, la boucle ne converge pas pour autant.

La prédiction théorique est nette : la frontière entre convergence et non-convergence est **exactement `delt = 2`**. On la mesure.

In [4]:
// Illustration 4 — Surface B : carte delt -> comportement de la boucle
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

var deltas = new double[] { 0.5, 1.0, 1.5, 1.9, 2.0, 2.1, 2.5, 3.0 };

Console.WriteLine(string.Format(CultureInfo.InvariantCulture, "{0,5}{1,10}{2,9}{3,10}  {4}",
    "delt", "L=|1-delt|", "a_120", "|a_120-b|", "comportement"));
Console.WriteLine(new string('-', 58));
foreach (double delt in deltas)
{
    var traj = Metajeux02Boucle.BoucleHoming(0.9, delt);
    string comp = Metajeux02Boucle.Classifie(traj);
    Console.WriteLine(string.Format(CultureInfo.InvariantCulture, "{0,5:F1}{1,10:F2}{2,9:F4}{3,10:F4}  {4}",
        delt,
        Math.Abs(1 - delt),
        traj[traj.Length - 1],
        Math.Abs(traj[traj.Length - 1] - Metajeux02Boucle.BCible),
        comp));
}

Console.WriteLine();
Console.WriteLine("Borne de Banach : contraction ssi L < 1 ssi delt dans (0, 2)");
Console.WriteLine("Frontiere mesuree : convergence pour delt <= 1.9, bascule a delt = 2.0 exactement.");

public static class Metajeux02Boucle
{
    public const double BCible = 0.3;

    // Clip sur le domaine [-1, 1].
    public static double Clip(double v) => Math.Max(-1.0, Math.Min(1.0, v));

    // Itère a' = clip(a + delt * (b - a)) et rend la trajectoire.
    public static double[] BoucleHoming(double a0, double delt, double b = 0.3, int n = 120)
    {
        var traj = new List<double> { a0 };
        double a = a0;
        for (int i = 0; i < n; i++)
        {
            a = Clip(a + delt * (b - a));
            traj.Add(a);
        }
        return traj.ToArray();
    }

    // converge / 2-cycle intérieur / 2-cycle sur rail, depuis la queue.
    public static string Classifie(double[] traj, double b = 0.3, double tol = 1e-6)
    {
        var queue = traj.Skip(Math.Max(0, traj.Length - 20)).ToArray();
        if (queue.All(v => Math.Abs(v - b) < 1e-4))
        {
            return "converge";
        }

        var ecarts = queue.Select(v => v - b).ToArray();
        bool pairesConstantes = true;
        for (int i = 0; i < ecarts.Length; i++)
        {
            double reference = (i % 2 == 0) ? ecarts[0] : ecarts[1];
            if (Math.Abs(ecarts[i] - reference) > tol)
            {
                pairesConstantes = false;
                break;
            }
        }

        if (pairesConstantes)
        {
            bool surRail = queue.Any(v => Math.Abs(Math.Abs(v) - 1.0) < 1e-9);
            return surRail ? "2-cycle sur rail" : "2-cycle interieur";
        }

        return "non stabilise";
    }
}

 deltL=|1-delt|    a_120 |a_120-b|  comportement


----------------------------------------------------------


  0.5      0.50   0.3000    0.0000  converge


  1.0      0.00   0.3000    0.0000  converge


  1.5      0.50   0.3000    0.0000  converge


  1.9      0.90   0.3000    0.0000  converge


... (2 outputs left out)

  2.5      1.50   1.0000    0.7000  2-cycle sur rail


  3.0      2.00   1.0000    0.7000  2-cycle sur rail


Borne de Banach : contraction ssi L < 1 ssi delt dans (0, 2)


Frontiere mesuree : convergence pour delt <= 1.9, bascule a delt = 2.0 exactement.


**Lecture de la sortie committée** : la frontière mesurée coïncide avec la borne théorique **exactement**. En dessous de `delt = 2` (`L < 1`), la boucle converge vers la cible `b = 0.3` — géométriquement vite près de `delt = 1` (`L = 0`, convergence en un pas), plus lentement aux bords. À `delt = 2.0` (`L = 1`, cas limite non contractant), la boucle rebondit : l'écart change de signe à chaque pas sans changer de magnitude — le **2-cycle exact** prédit par l'analyse. Au-delà (`L > 1`), l'écart s'amplifie à chaque pas jusqu'à mordre le rail `|a| = 1` ; le clip a alors remplacé la divergence par un **2-cycle sur rail** — une des deux valeurs du cycle est collée au mur — la boucle bornée n'est plus pour autant convergente. Le clip ne répare pas la perte de contraction : il en masque la conséquence visible.

## 4. Le régime exact de la boucle GT-18 — jamais contractante

Reconsidérons la boucle effectivement committée dans GT-18 : `a' = clip(a + delta, -1, 1)` — un pas **additif** constant. Sa pente est `1` partout où le clip ne mord pas : constante de Lipschitz `L = 1`, **le cas limite non contractant, pour toute valeur de `delta`**. Banach ne s'applique pas ; ce qui fait converger la boucle de GT-18 vers `1.0` (cas `delta = 0.5`) est uniquement le **rail** : chaque pas ajoute `delta`, le clip tronque à `1`, et la valeur s'y colle — c'est une convergence-au-rail en temps borné (au plus `2/delta` pas), pas une convergence géométrique vers un point fixe intérieur. La distinction n'est pas cosmétique : la convergence-au-rail exige un mur pour s'arrêter, la convergence de Banach non — la surface B vient de le mesurer (`delt > 2` : même avec un rail, pas d'arrêt).

In [5]:
// Illustration 5 — GT-18 reframed : la boucle additive est le cas L=1 partout
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

Console.WriteLine("Boucle additive GT-18 (L=1 partout, jamais contractante) :");
foreach (double delta in new double[] { 0.5, -0.3 })
{
    var traj = Metajeux02BoucleAdditive.BoucleAdditive(0.5, delta);
    double rail = delta > 0 ? 1.0 : -1.0;
    int? nRail = null;
    for (int i = 0; i < traj.Length; i++)
    {
        if (Math.Abs(traj[i] - rail) < 1e-12)
        {
            nRail = i;
            break;
        }
    }
    Console.WriteLine(string.Format(CultureInfo.InvariantCulture,
        "  delta={0:+0.0;-0.0} : atteint le rail en {1} pas, y reste (a_120 = {2:+0.0;-0.0})",
        delta, nRail, traj[traj.Length - 1]));
}

Console.WriteLine();
Console.WriteLine("Boucle homing contractante (delt=0.8, L=0.2) : convergence GEOMETRIQUE interieure");
var trajHoming = Metajeux02BoucleAdditive.BoucleHoming(0.9, 0.8);
var errs = trajHoming.Take(8).Select(a => Math.Abs(a - 0.3)).ToArray();
Console.WriteLine($"  |a_n - b| pour n=0..7 : [{string.Join(", ", errs.Select(e => e.ToString("0.0000", CultureInfo.InvariantCulture)))}]");
double ratio = Math.Pow(errs[errs.Length - 1] / errs[0], 1.0 / 7.0);
Console.WriteLine($"  ratio moyen d'amortissement par pas : {ratio.ToString("0.000", CultureInfo.InvariantCulture)} (theorie L = 0.2)");
Console.WriteLine();
Console.WriteLine("Convergence-au-rail : temps borne par le mur, arret PAR le mur.");
Console.WriteLine("Convergence de Banach : taux geometrique |1-delt|, arret par amortissement interne.");

public static class Metajeux02BoucleAdditive
{
    public static double Clip(double v) => Math.Max(-1.0, Math.Min(1.0, v));

    // Boucle GT-18 exacte : a' = clip(a + delta).
    public static double[] BoucleAdditive(double a0, double delta, int n = 120)
    {
        var traj = new List<double> { a0 };
        double a = a0;
        for (int i = 0; i < n; i++)
        {
            a = Clip(a + delta);
            traj.Add(a);
        }
        return traj.ToArray();
    }

    // Itère a' = clip(a + delt * (b - a)) et rend la trajectoire.
    public static double[] BoucleHoming(double a0, double delt, double b = 0.3, int n = 120)
    {
        var traj = new List<double> { a0 };
        double a = a0;
        for (int i = 0; i < n; i++)
        {
            a = Clip(a + delt * (b - a));
            traj.Add(a);
        }
        return traj.ToArray();
    }
}

Boucle additive GT-18 (L=1 partout, jamais contractante) :


  delta=+0.5 : atteint le rail en 1 pas, y reste (a_120 = +1.0)


  delta=-0.3 : atteint le rail en 5 pas, y reste (a_120 = -1.0)


Boucle homing contractante (delt=0.8, L=0.2) : convergence GEOMETRIQUE interieure


  |a_n - b| pour n=0..7 : [0.6000, 0.1200, 0.0240, 0.0048, 0.0010, 0.0002, 0.0000, 0.0000]


  ratio moyen d'amortissement par pas : 0.200 (theorie L = 0.2)


Convergence-au-rail : temps borne par le mur, arret PAR le mur.


Convergence de Banach : taux geometrique |1-delt|, arret par amortissement interne.


**Lecture de la sortie committée** : les deux dynamiques sont distinguées par leur signature. La boucle additive de GT-18 atteint le rail en un nombre entier de pas (`2/delta` ici : 1 pas pour `delta = 0.5` partant de 0.5) et s'y colle — l'arrêt est un événement *discret*, causé par le mur. La boucle homing contractante, elle, amortit son erreur d'un facteur `L` à chaque pas — mesuré `≈ 0.2`, théorie `0.2` — et l'arrêt est *interieur*, asymptotique, sans mur. Le jouet GT-18 vivait donc dans le cas dégénéré `L = 1` : sa « convergence vers le point fixe » était la sédimentation sur le rail, et la section 3 de GT-18 mesurait la distance à un « point fixe » qui n'en était un que par clip.

## 5. Verdict par surface — ce qui cède, ce qui tient

| Surface | Ce qui cède | Ce qui tient | Condition exacte |
|---|---|---|---|
| **A — sélection compositionnelle** | composer des *sélections* d'argmax locaux sous plateau multivoque + couplage : le profil `(0,0)` prédit n'est pas Nash (`écart +0.300`) | la propriété de Hedges au sens des **correspondances** (l'ensemble composé sur-approxime les équilibres : `(1,1)` y est) ; la composition des sélections en régime **univoque** (section 1) | la casse exige la conjonction plateau × couplage — chacun seul est inoffensif (section 1, contre-fait r = 0) |
| **B — convergence de la boucle** | la lecture « la boucle converge donc elle est saine » : à `L ≥ 1` elle ne converge pas (2-cycle intérieur à `delt = 2`, 2-cycle sur rail au-delà) ; la boucle GT-18 est `L = 1` **partout** — jamais contractante | la frontière de contraction est **exactement** la borne de Banach `delt = 2` — le théorique et le mesuré coïncident sans zone grise | convergence géométrique ssi `delt ∈ (0, 2)` ; convergence-au-rail de GT-18 = arrêt par le mur, pas par amortissement |

Le bilan de maturation est positif au sens strict : le jouet GT-18 **survit borné** — aucun de ses résultats numériques n'était faux — mais son périmètre de validité est désormais écrit : la composition des sélections exige l'univocité (ou l'absence de couplage), et la convergence de sa boucle est de type rail, pas de type Banach. C'est ce que « survivre à une variante -b » veut dire : pas rester intact, avoir ses bornes.

### Questions ouvertes — ce que cette variante ne tranche pas

- **La sélection d'équilibre en amont.** La surface A montre que la composition sur-approxime l'ensemble des équilibres ; elle ne dit pas lequel un processus décentralisé sélectionne — ni si un raffinement (élimination des équilibres faibles par indifférence) suffirait à rétablir une composition des sélections *strictes*.
- **Le plateau exact est une symétrie.** La multivocité exhibée naît de `c = 0.5`, milieu exact de deux actions — un plateau de mesure nulle dans l'espace des paramètres. Un plateau épais (plateau d'utilité flat sur un intervalle, fréquent en jeux discrets de coordination) casserait de la même façon, mais la frontière « plateau de mesure nulle vs épais » n'est pas explorée ici.
- **Le clip lisse-t-il jamais ?** Sur la surface B, le clip a borné la divergence en oscillation. Il existe des update non lisses où le clip produit au contraire des points fixes attractifs *sur* le rail (convergence vers le mur avec approche unilatérale) — la taxonomy complète des régimes `L > 1` bornés n'est pas dressée.

## Exercice 1 : le seuil de couplage

La casse de la surface A exige `r > 0` — mais à quel seuil exactement ? Le profil `(0, 0)` prédit par la composition naïve cesse d'être un équilibre dès que `r` est strictement positif. Vérifiez-le : balayez `r` sur `Enumerable.Range(0, 7).Select(i => i * 0.1)` et, pour chaque valeur, recalculez l'écart de déviation de J1 au profil `(0, 0)`. Confirmez que l'écart vaut exactement `r` (pourquoi ?) et dites ce que devient le verdict à `r = 0`.

In [6]:
// Exercice 1 à compléter
using System;
using System.Linq;

int[] ActionsXEx1 = { 0, 1, 2 };
int[] ActionsYEx1 = { 0, 1, 2 };
const double CPlateauEx1 = 0.5;

double U2Ex1(double x, double y) => -(y - x) * (y - x);
int Br2Ex1(double x) => ActionsYEx1.OrderByDescending(y => U2Ex1(x, y)).First();

// TODO etudiant : definir u1_A(r, x, y) = -(x - 0.5)^2 + r * y

// 1. Pour r dans Enumerable.Range(0, 7).Select(i => i * 0.1) : reconstruire u1_A avec ce couplage.
double[] CouplagesEx1 = Enumerable.Range(0, 7).Select(i => i * 0.1).ToArray();

// 2. TODO etudiant : calculer l'ecart de deviation de J1 au profil (0, 0) :
//    max_x u1_A(x, br2(x)) - u1_A(0, 0)
// 3. TODO etudiant : verifier que l'ecart == r exactement, et expliquer pourquoi
// Indice : -(1-0.5)^2 + r*1 - (-(0-0.5)^2 + r*0) = 0 + r = r

Console.WriteLine("Exercice a completer");

// TODO etudiant : remplacer null par le tableau des ecarts mesures
double[] resultat_ex1 = null;

Console.WriteLine($"Resultat ex1 (a completer) : {(resultat_ex1 is null ? "aucune valeur" : string.Join(", ", resultat_ex1))}");

Exercice a completer


Resultat ex1 (a completer) : aucune valeur


## Exercice 2 : le 2-cycle en main

À `delt = 2.0` exactement, la théorie prédit que l'écart `a - b` change de signe à chaque pas sans changer de magnitude : `a_{n+1} - b = -(a_n - b)`. Partant de `a0 = 0.9` avec `b = 0.3` : calculez à la main les quatre premiers termes de la suite des écarts, puis vérifiez avec `boucle_homing` que la trajectoire exacte les reproduit. Combien vaut l'amplitude du cycle, et pourquoi ne décroît-elle jamais ?

In [7]:
// Exercice 2 à compléter
using System;
using System.Collections.Generic;

const double BCibleEx2 = 0.3;
const double A0Ex2 = 0.9;
const double DeltEx2 = 2.0;

// 1. Main : a0 = 0.9, b = 0.3, delt = 2.0 -> ecarts a_n - b pour n = 0..3
// TODO etudiant : ecrire a la main les quatre premiers ecarts theoriques

// 2. TODO etudiant : verifier avec BoucleHomingEx2(0.9, 2.0) que les 4 premiers ecarts sont reproduits
// 3. TODO etudiant : amplitude du cycle = |a0 - b| : pourquoi constante ?
// Indice : a' - b = clip(a + 2(b-a)) - b = (a + 2b - 2a) - b = -(a - b) tant que le clip ne mord pas

Console.WriteLine("Exercice a completer");

// TODO etudiant : remplacer null par les ecarts theoriques (n = 0..3)
double[] resultat_ex2 = null;

Console.WriteLine($"Resultat ex2 (a completer) : {(resultat_ex2 is null ? "aucune valeur" : string.Join(", ", resultat_ex2))}");

public static class Metajeux02Ex2
{
    public static double Clip(double v) => Math.Max(-1.0, Math.Min(1.0, v));

    // Itère a' = clip(a + delt * (b - a)) et rend la trajectoire.
    public static double[] BoucleHomingEx2(double a0, double delt, double b = 0.3, int n = 120)
    {
        var traj = new List<double> { a0 };
        double a = a0;
        for (int i = 0; i < n; i++)
        {
            a = Clip(a + delt * (b - a));
            traj.Add(a);
        }
        return traj.ToArray();
    }
}

Exercice a completer


Resultat ex2 (a completer) : aucune valeur


## Exercice 3 : la cible hors du domaine

Toute la surface B vise `b = 0.3`, intérieur au domaine `[-1, 1]` du clip. Que se passe-t-il quand la cible est **hors** du domaine — `b = 1.5` ? Prédisez d'abord (par écrit) : la boucle `a' = clip(a + delt*(b-a))` avec `delt = 0.8` converge-t-elle, et vers quoi ? Vérifiez numériquement : la limite est-elle le point fixe de Banach, ou autre chose ? Mesurez la distance finale et interprétez : le clip **projette** le point fixe sur le domaine fermé.

In [8]:
// Exercice 3 à compléter
using System;
using System.Collections.Generic;

// 1. Prediction ecrite AVANT simulation : convergence vers b=1.5 ? vers 1.0 ? autre ?
// TODO etudiant : ecrire votre prediction en commentaire, puis la verifier

// 2. TODO etudiant : simuler BoucleHomingEx3(0.9, 0.8, b=1.5, n=120) et mesurer la limite
// 3. TODO etudiant : distance finale : |limite - 1.5| = 0.5 -> le point fixe atteignable est projete sur le bord
// Indice : si a < 1 alors a' = a + 0.8*(1.5-a) > a (strictement) et clip(a') = 1 des que a >= 1

Console.WriteLine("Exercice a completer");

// TODO etudiant : remplacer null par la limite mesuree
double? resultat_ex3 = null;

Console.WriteLine($"Resultat ex3 (a completer) : {(resultat_ex3 is null ? "aucune valeur" : resultat_ex3.Value.ToString("0.0000"))}");

public static class Metajeux02Ex3
{
    public static double Clip(double v) => Math.Max(-1.0, Math.Min(1.0, v));

    // Itère a' = clip(a + delt * (b - a)) et rend la trajectoire.
    public static double[] BoucleHomingEx3(double a0, double delt, double b = 0.3, int n = 120)
    {
        var traj = new List<double> { a0 };
        double a = a0;
        for (int i = 0; i < n; i++)
        {
            a = Clip(a + delt * (b - a));
            traj.Add(a);
        }
        return traj.ToArray();
    }
}

Exercice a completer


Resultat ex3 (a completer) : aucune valeur


## Conclusion et perspectives

GT-18 a survécu à sa variante -b au sens de la maturation : ses résultats tiennent, **et leurs bornes sont écrites**. Trois acquis :

1. **La sélection n'est pas compositionnelle** — la composition de sélections d'argmax locaux rend un profil non-Nash (`(0, 0)`, écart `+0.300`) dès qu'un plateau multivoque rencontre un couplage ; la propriété de Hedges n'a jamais promis autre chose que la composition des *correspondances*, qui sur-approxime l'ensemble des équilibres. Prédire le point exige l'univocité — ou une théorie de la sélection.
2. **La frontière de contraction est la borne de Banach, exactement** — mesurée à `delt = 2` sans zone grise ; au-delà, le clip borne la divergence en oscillation sur rail, il ne la répare pas.
3. **La boucle GT-18 est le cas dégénéré `L = 1`** — sa convergence est une convergence-au-rail (arrêt par le mur, temps borné), pas une convergence géométrique de Banach (arrêt par amortissement interne). Les deux signatures sont mesurées et distinguées.

Perspectives : le même patron s'applique à chaque objet mathématique porté par un jouet — vérifier la propriété sur le régime où elle tient (GT-18 l'a fait), puis chercher la frontière où elle cesse (ce notebook), puis écrire les bornes dans la prose (fait au §5). Le voisinage immédiat : la sélection d'équilibre dans les jeux composés à contextes multiples, et la taxonomy complète des boucles bornées non contractantes.

## References

- **Ghani, N., Hedges, J., Winschel, V., Zahn, P. (2018).** "Compositional Game Theory." *Proceedings of the 33rd Annual ACM/IEEE Symposium on Logic in Computer Science (LICS '18)*, 472-481, DOI 10.1145/3209108.3209165. — formalise la composition séquentielle et parallèle de jeux dans un cadre catégorique ; fondation de la lecture compositionnelle des correspondances d'équilibre.
- **Sandholm, W. H. (2010).** *Population Games and Evolutionary Dynamics.* MIT Press. — dynamique de population et evolution des équilibres, base théorique des modèles d'équilibre évolutionnaire.
- **Kakutani, S. (1941).** "A generalization of Brouwer's fixed-point theorem." *Duke Mathematical Journal* 8(3), 457-459. — théorème fondateur des correspondances multivoques, base de Nash 1950 et de la multivocité exhibée en Surface A.
- **Govindan, S., Wilson, R. (2003).** "A global Newton method to compute Nash equilibria." *Journal of Economic Theory* 110(1), 65-86, DOI 10.1016/S0022-0531(03)00005-X. — méthode numérique de calcul d'équilibres par homotopie.
- **Capucci, M., Gavranović, B., Hedges, J., Rischel, E. F. (2022).** "Towards Foundations of Categorical Cybernetics." *Electronic Proceedings in Theoretical Computer Science* 372, 235-248, DOI 10.4204/EPTCS.372.17. — lenses et composition de systèmes dynamiques, prolonge la composition Ghani-Hedges-Winschel-Zahn 2018 aux boucles de rétroaction comme celle de GT-18.
- **Bolt, J., Hedges, J., Zahn, P. (2019).** "Bayesian open games." *Compositionality* 5, article 9, 2023, arXiv:1910.03656, DOI 10.32408/compositionality-5-9. — la composition des *correspondances* d'équilibre, que la Conclusion identifie comme source de la sur-approximation en Surface A.

## À retenir

- **La sélection d'équilibre n'est PAS compositionnelle.** Composer des sélections d'argmax locaux peut produire un profil non-Nash : sur la surface A, le profil `(0, 0)` prédit par la composition naïve cesse d'être un équilibre dès qu'un plateau multivoque rencontre un couplage (écart de déviation `+0.300`).
- **La propriété de Hedges promet la composition des *correspondances*, pas des *sélections*.** Elle sur-approxime l'ensemble des équilibres ; prédire le point exact exige l'univocité — ou une théorie de la sélection.
- **La frontière de contraction est la borne de Banach, exactement** : mesurée à `delt = 2` sans zone grise. Au-delà, le clip borne la divergence en oscillation sur le rail, il ne la répare pas. La boucle GT-18 est le cas dégénéré `L = 1` (convergence-au-rail par le mur, temps borné), distincte de la convergence géométrique de Banach.

## Vérifiez votre compréhension

1. Pourquoi la composition sur-approxime-t-elle l'ensemble des équilibres sans sélectionner l'un d'eux ? Quel est le lien avec le plateau `c = 0.5` ?
2. À `delt = 2` exactement, que prédit la théorie pour la suite des écarts `a_n − b` ? Pourquoi l'amplitude du cycle ne décroît-elle jamais ?
3. En quoi la convergence de la boucle GT-18 est-elle une « convergence-au-rail » et non une convergence géométrique de Banach ? Comment les deux signatures sont-elles distinguées ?

<details>
<summary>Éléments de réponse</summary>

1. La composition des correspondances transporte tous les profils admissibles : elle rend l'ensemble `[(0, 0), (1, 1)]` (cellule 6), sans aucun critère pour préférer l'un des deux. Le plateau `c = 0.5` est exactement ce qui produit cette multivocité : les valeurs locales de J1 sont `-0.250, -0.250, -2.250`, donc `x = 0` et `x = 1` sont à égalité stricte et `BR = {0, 1}` ; la correspondance composée contient les deux profils, elle n'en sélectionne aucun. La sélection est ajoutée par la règle « plus petit argmax », et c'est elle que le jeu composé réfute : le profil `(0, 0)` a un écart de déviation `+0.300` (« NON-NASH du compose »), tandis que `(1, 1)` a `+0.000` (« equilibre du compose »). Sans couplage (`r = 0`), les écarts valent `+0.000, +0.000` : le plateau seul est inoffensif, c'est le retour `r * y` qui rend le choix de `x` payant et fait de l'ensemble une sur-approximation (cellule 8).

2. À `delt = 2`, `L = |1 - delt| = 1` : l'application n'est ni contractante ni dilatante. Tant que le clip ne mord pas, `a_{n+1} - b = -(a_n - b)` (indice de l'exercice 2), donc la suite des écarts alterne de signe à amplitude constante. L'amplitude ne décroît jamais parce que `|L| = 1` exactement : chaque pas est une réflexion exacte autour de `b`, il n'y a ni amortissement (`L < 1`) ni amplification (`L > 1`). La mesure le confirme à `delt = 2.0` : `L = 1.00`, `a_120 = 0.9000`, `|a_120 - b| = 0.6000`, classé « 2-cycle interieur » (cellule 11), la valeur `0.6` étant l'écart initial `0.9 - 0.3`.

3. La boucle additive de GT-18 (`a' = clip(a + delta)`) a `L = 1` partout et n'est jamais contractante : elle atteint le rail en un temps borné par le mur (1 pas pour `delta = +0.5`, 5 pas pour `delta = -0.3`) et y reste, l'arrêt étant causé par le clip et non par un amortissement interne (cellule 14). Une convergence de Banach a au contraire un taux `|1 - delt| < 1` et un écart qui décroît géométriquement : pour `delt = 0.8` (`L = 0.2`), les écarts mesurés sont `0.6000, 0.1200, 0.0240, 0.0048, 0.0010, 0.0002, 0.0000, 0.0000` avec un ratio moyen de `0.200`, égal à la théorie. Les deux signatures se distinguent donc sur trois traits mesurés dans la cellule 14 : le point d'arrêt (rail `|a| = 1` contre point intérieur `b`), la forme de la suite des écarts (amplitude constante contre décroissance géométrique de taux `|1 - delt|`), et la cause de l'arrêt (le mur contre l'amortissement interne).

</details>

## Pour aller plus loin

- **Sélection d'équilibre dans les jeux composés** à contextes multiples : le voisinage immédiat de ce notebook.
- **Taxonomie des boucles bornées non contractantes** : classifier tous les régimes `L > 1` bornés (le clip borne-t-il toujours en oscillation ?).
- **Kakutani (1941)** — le théorème des correspondances multivoques, base de la multivocité exhibée en surface A.

> Carnet issu de l'EPIC #20250 (jumeau C# de `Metajeux-02-Casser-la-Composition-Python.ipynb`, grain de parité C# #26534).